In [1]:
import numpy as np 
import pandas as pd 

In [5]:
DATA_PATH = "../data/transactions.parquet"

In [6]:
df =pd.read_parquet(DATA_PATH)

In [12]:
print("Shape:", df.shape)
display(df.head())
df.info()

Shape: (80000, 20)


,transaction_id,transaction_time,amount,merchant_category,device_type,new_device,international,card_present,distance_from_home_km,transactions_last_10m,failed_attempts_24h,account_age_days,avg_amount_30d,amount_to_avg_ratio,ip_risk_score,merchant_risk_score,hour,day_of_week,is_weekend,is_fraud
0,TX-000000001,2025-01-01 00:00:13,70.897097,fuel,pos,1,0,1,5.497668,0,1,1692.886607,36.654755,1.934186,0.293980,0.084712,0,2,0,0
1,TX-000000002,2025-01-01 00:00:46,24.839975,restaurant,atm,0,0,1,8.424386,0,0,2345.388085,20.194129,1.230059,0.029912,0.220515,0,2,0,0
2,TX-000000003,2025-01-01 00:03:24,47.136527,digital_goods,mobile,0,0,0,14.728261,0,0,1444.831389,52.326798,0.900810,0.180774,0.062035,0,2,0,0
3,TX-000000004,2025-01-01 00:04:46,27.214357,grocery,atm,0,0,1,22.927283,0,1,1688.291552,41.665158,0.653168,0.060403,0.152603,0,2,0,0
4,TX-000000005,2025-01-01 00:04:59,25.527235,restaurant,pos,0,0,1,11.617231,0,0,476.013104,144.499259,0.176660,0.088575,0.131824,0,2,0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80000 entries, 0 to 79999
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   transaction_id         80000 non-null  object        
 1   transaction_time       80000 non-null  datetime64[ns]
 2   amount                 80000 non-null  float64       
 3   merchant_category      80000 non-null  object        
 4   device_type            80000 non-null  object        
 5   new_device             80000 non-null  int64         
 6   international          80000 non-null  int64         
 7   card_present           80000 non-null  int64         
 8   distance_from_home_km  80000 non-null  float64       
 9   transactions_last_10m  80000 non-null  int64         
 10  failed_attempts_24h    80000 non-null  int64         
 11  account_age_days       80000 non-null  float64       
 12  avg_amount_30d         80000 non-null  float64       
 13  a

# Target found , `is_fraud` found 

In [13]:
print(df.columns.tolist())

['transaction_id', 'transaction_time', 'amount', 'merchant_category', 'device_type', 'new_device', 'international', 'card_present', 'distance_from_home_km', 'transactions_last_10m', 'failed_attempts_24h', 'account_age_days', 'avg_amount_30d', 'amount_to_avg_ratio', 'ip_risk_score', 'merchant_risk_score', 'hour', 'day_of_week', 'is_weekend', 'is_fraud']


# Target validation 
- target only should either 0 or 1
- target shoud not containe other than 0 and 1
- find ration of positive and negative

In [44]:
target =df["is_fraud"].value_counts(dropna=False)
print(f'Distribution checking {target}')
print("\n\n")
print(target.index.tolist())
if target.shape[0] <= 2 and set(target.index).issubset({1, 0}):
    all_positive = target.get(1, 0)  # Safely gets count for 1, defaults to 0 if missing
    all_negative = target.get(0, 0)  # Safely gets count for 0, defaults to 0 if missing
    
    total = len(df)
    
    print(f'Calculate fraud rate/ Fradulent % in data: {(all_positive / total) * 100:.2f}%')
    print(f'Calculate ligitimate % in data: {(all_negative / total) * 100:.2f}%')
else: 
    print("Something wrong, please check the dataset (contains NaNs or non-binary values)")
fraud_rate = df["is_fraud"].mean() * 100



Distribution checking is_fraud
0    79840
1      160
Name: count, dtype: int64



[0, 1]
Calculate fraud rate/ Fradulent % in data: 0.20%
Calculate ligitimate % in data: 99.80%


## Validation summery 

In [48]:
validation_report = {
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_rows": int(df.duplicated().sum()),
    "null_target": int(df["is_fraud"].isna().sum()),
    "unique_target_values": df["is_fraud"].unique().tolist(),
}

validation_report

{'rows': 80000,
 'columns': 20,
 'duplicate_rows': 0,
 'null_target': 0,
 'unique_target_values': [0, 1]}

In [49]:
df.columns.tolist()

['transaction_id',
 'transaction_time',
 'amount',
 'merchant_category',
 'device_type',
 'new_device',
 'international',
 'card_present',
 'distance_from_home_km',
 'transactions_last_10m',
 'failed_attempts_24h',
 'account_age_days',
 'avg_amount_30d',
 'amount_to_avg_ratio',
 'ip_risk_score',
 'merchant_risk_score',
 'hour',
 'day_of_week',
 'is_weekend',
 'is_fraud']

In [50]:
df['transaction_time']

0       2025-01-01 00:00:13
1       2025-01-01 00:00:46
2       2025-01-01 00:03:24
3       2025-01-01 00:04:46
4       2025-01-01 00:04:59
                ...        
79995   2025-06-29 23:49:49
79996   2025-06-29 23:53:30
79997   2025-06-29 23:54:02
79998   2025-06-29 23:55:27
79999   2025-06-29 23:57:17
Name: transaction_time, Length: 80000, dtype: datetime64[ns]

In [51]:
dft=df.copy()


In [56]:
dft["transaction_time"] = pd.to_datetime(
    dft["transaction_time"],
    errors="coerce"
)



In [61]:
# Cyclical encoding
dft["hour_sin"] = np.sin(2 * np.pi * dft["transaction_time"].dt.hour / 24)
dft["hour_cos"] = np.cos(2 * np.pi * dft["transaction_time"].dt.hour / 24)

In [65]:
dft["hour"] = dft["transaction_time"].dt.hour.astype("category")


In [66]:
dft.columns.tolist()

['transaction_id',
 'transaction_time',
 'amount',
 'merchant_category',
 'device_type',
 'new_device',
 'international',
 'card_present',
 'distance_from_home_km',
 'transactions_last_10m',
 'failed_attempts_24h',
 'account_age_days',
 'avg_amount_30d',
 'amount_to_avg_ratio',
 'ip_risk_score',
 'merchant_risk_score',
 'hour',
 'day_of_week',
 'is_weekend',
 'is_fraud',
 'hour_sin',
 'hour_cos']

In [67]:
dft['hour']

0         0
1         0
2         0
3         0
4         0
         ..
79995    23
79996    23
79997    23
79998    23
79999    23
Name: hour, Length: 80000, dtype: category
Categories (24, int32): [0, 1, 2, 3, ..., 20, 21, 22, 23]